# 🎤 VoiceOnly — Google Colab 에디션

**원곡에서 이미 녹음된 보컬과 반주를 분리**하는 개인용 도구야. AI로 새로운 가수의 목소리를 생성하거나 복제하지 않아.

**사용법:** 위쪽 메뉴에서 **런타임 → 모두 실행**을 선택하거나, 아래 코드 셀들의 ▶ 버튼을 순서대로 눌러줘. 2번 단계에서 음악 파일을 하나 선택하면, 보컬·반주 분리 후 ZIP 파일을 받을 수 있어.

- GPU를 사용할 수 있으면 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택해줘. 무료 Colab에서는 GPU를 사용하지 못할 수도 있어. CPU도 동작하지만 오래 걸릴 수 있어.
- **음원 업로드는 이 Colab 안에서만** 진행돼. GitHub Pages는 안내와 실행 링크만 제공해.
- Colab의 임시 실행 환경에 파일이 업로드되므로 민감한 녹음은 주의해. 세션이 종료되면 임시 파일은 사라질 수 있어.
- 모델을 처음 사용할 때 다운로드가 필요하고, 보컬에 악기 소리가 조금 남을 수도 있어.
- 저작권이 있는 음악의 공개 배포·리믹스에는 관련 권리자의 허락이 필요할 수 있어.

🔗 [VoiceOnly GitHub 저장소](https://github.com/oddpennine/voiceonly)


In [ ]:
# 1. 프로그램 준비 — 첫 실행에만 시간이 걸릴 수 있어
import subprocess, sys, shutil

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    print("FFmpeg 설치 중…")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

print("Demucs 설치 중…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs>=4,<5"], check=True)
import torch
print("✅ 준비 완료 / 실행 장치:", "GPU" if torch.cuda.is_available() else "CPU")


In [ ]:
# 2. 원본 음원 업로드 — 한 번에 하나만 선택해줘
from google.colab import files
from pathlib import Path
import uuid

allowed = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".aac", ".aiff"}
print("🎵 MP3 / WAV / FLAC / M4A 등의 음원을 하나 선택해줘.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("음원 파일 하나만 업로드해줘. 이 셀을 다시 실행하면 돼.")
name, audio_bytes = next(iter(uploaded.items()))
extension = Path(name).suffix.lower()
if extension not in allowed:
    raise ValueError(f"지원하지 않는 음원 형식이야: {extension or '(확장자 없음)'}")
if not audio_bytes:
    raise ValueError("빈 파일은 분리할 수 없어.")
input_dir = Path("/content/voiceonly_input")
input_dir.mkdir(parents=True, exist_ok=True)
source = input_dir / ("song_" + uuid.uuid4().hex[:10] + extension)
source.write_bytes(audio_bytes)
print(f"✅ 업로드 완료: {name} ({len(audio_bytes) / 1048576:.1f} MB)")


In [ ]:
# 3. Demucs로 보컬/반주 분리 — 원곡 전체 처리
# 'htdemucs_ft'로 바꾸면 더 느리지만 분리 품질이 개선될 수도 있어.
# 런타임 GPU 메모리 부족 시 아래 DEVICE를 'cpu'로 변경해줘.
MODEL = "htdemucs"    # 또는 'htdemucs_ft'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import subprocess, sys
from pathlib import Path
output_dir = Path("/content/voiceonly_separated")
output_dir.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, "-m", "demucs", "--two-stems=vocals",
    "-n", MODEL, "-d", DEVICE, "-o", str(output_dir), str(source)
]
print(f"🎧 분리 중… 모델: {MODEL} / 장치: {DEVICE}. 긴 곡은 시간이 걸릴 수 있어.", flush=True)
try:
    subprocess.run(command, check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError("분리에 실패했어. 위 오류 로그를 확인해줘. GPU 메모리 부족이면 CPU로 바꿔봐.") from exc

stem_dir = output_dir / MODEL / source.stem
vocals_wav = stem_dir / "vocals.wav"
instrumental_wav = stem_dir / "no_vocals.wav"
if not vocals_wav.is_file() or not instrumental_wav.is_file():
    raise FileNotFoundError("분리 결과를 못 찾았어. 모델 실행 로그를 확인해줘.")
print("✅ 분리 완료!", vocals_wav.name, instrumental_wav.name)


In [ ]:
# 4. MP3 생성 + 미리 듣기 — WAV 원본도 유지돼
import subprocess
from IPython.display import display, Audio, Markdown

MP3_BITRATE = "192k"  # 예: '256k', '320k'로 변경 가능

def wav_to_mp3(wav_path, mp3_path):
    subprocess.run([
        "ffmpeg", "-y", "-nostdin", "-hide_banner", "-loglevel", "error",
        "-i", str(wav_path), "-c:a", "libmp3lame", "-b:a", MP3_BITRATE, str(mp3_path)
    ], check=True)

vocals_mp3 = stem_dir / "vocals.mp3"
instrumental_mp3 = stem_dir / "instrumental.mp3"
wav_to_mp3(vocals_wav, vocals_mp3)
wav_to_mp3(instrumental_wav, instrumental_mp3)

print("🎤 보컬 미리 듣기")
display(Audio(filename=str(vocals_mp3)))
print("🎹 반주 미리 듣기")
display(Audio(filename=str(instrumental_mp3)))


In [ ]:
# 5. 결과 다운로드 — 한 개의 ZIP 파일로 내려받기
# True이면 원본 분리 결과 WAV까지 함께 저장해. ZIP 파일 용량이 커질 수 있어.
INCLUDE_WAV = True

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

archive = Path("/content/VoiceOnly_vocals_and_instrumental.zip")
results = [
    (vocals_mp3, "vocals.mp3"),
    (instrumental_mp3, "instrumental.mp3"),
]
if INCLUDE_WAV:
    results.extend([
        (vocals_wav, "vocals.wav"),
        (instrumental_wav, "instrumental.wav"),
    ])

with ZipFile(archive, "w", compression=ZIP_DEFLATED, compresslevel=3) as bundle:
    for path, archive_name in results:
        if not path.is_file():
            raise FileNotFoundError(f"결과 파일을 찾지 못했어: {path}")
        bundle.write(path, arcname=archive_name)

print("✅ 압축 완료:", ", ".join(name for _, name in results))
print(f"ZIP 용량: {archive.stat().st_size / 1048576:.1f} MB")
print("다운로드가 자동 시작되지 않으면 이 셀을 다시 실행해줘.")
files.download(str(archive))


## 문제 해결

- **Demucs 설치 오류:** 런타임을 새로 시작한 다음 1번 셀부터 다시 실행해줘. Colab·패키지 환경 변경에 따라 추가 수정이 필요할 수 있어.
- **GPU 메모리 부족:** 3번 코드의 `DEVICE = "cpu"`로 바꿔 실행해줘.
- **결과에 악기가 들림:** 3번 코드의 `MODEL = "htdemucs_ft"`로 바꿔 시도해봐. 완전한 무반주를 보장하진 않아.
- **파일 다운로드가 너무 큼:** 5번 코드의 `INCLUDE_WAV = False`로 변경하면 MP3 파일만 ZIP에 들어가.
- **대용량 업로드가 멈춤:** 브라우저 업로드가 제한될 수 있어. 파일을 줄이거나 Google Drive에서 직접 불러오는 별도 방식을 사용해야 해.
